## Appendix

In [1]:
from scipy.stats import chi2
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
import numpy as np
import pandas as pd
from itertools import product

pd.set_option("display.width", 2000)

# import custom functions
from ST5213_helpers import *

### Task 1

#### (a)

In [2]:
print("n1+: ", 195+46)
print("n2+: ", 979+370)
print("n+1: ", 195+979)
print("n+2: ", 46+370)
print("n++: ", 195+46+979+370)

n1+:  241
n2+:  1349
n+1:  1174
n+2:  416
n++:  1590


In [3]:
M = 1590
n = 241
N = 1174

p195 = stats.hypergeom.pmf(195, M=M, n=n, N=N)
print("When N11=195: ", p195)
p_list = stats.hypergeom.pmf(range(0, 242), M=M, n=n, N=N)
p_val = p_list[p_list<=p195].sum()
print("P-value: ", p_val)

When N11=195:  0.001423480678571258
P-value:  0.006722755222685084


#### (b)

##### (i)

In [4]:
# White children using Pearson chi-square
nij = np.array([[104, 10], [22, 2]])
print(nij)
muhat = np.outer(nij.sum(axis=1), nij.sum(axis=0)) / nij.sum()
print(muhat)

X2 = ((nij - muhat)**2 / muhat).sum()
print("Pearson chi-square: ", X2, chi2.sf(X2, 1))

[[104  10]
 [ 22   2]]
[[104.08695652   9.91304348]
 [ 21.91304348   2.08695652]]
Pearson chi-square:  0.004803675856307418 0.944744015864725


##### (ii)

In [5]:
# Black children using likelihood ratio test
nij = np.array([[91, 36], [957, 368]])
print(nij)
muhat = np.outer(nij.sum(axis=1), nij.sum(axis=0)) / nij.sum()
print(muhat)

G2 = 2 * (nij * np.log(nij/muhat)).sum()
print("Likelihood ratio test: ", G2, chi2.sf(G2, 1))

[[ 91  36]
 [957 368]]
[[ 91.66391185  35.33608815]
 [956.33608815 368.66391185]]
Likelihood ratio test:  0.01887392798731069 0.8907285299418354


### Task 2

### (a)

In [6]:
df = pd.DataFrame(list(product(['Both parents', 'Mother only'], ['B', 'W'], ['M', 'F'], [1, 2], ['N', 'S'])), 
                  columns=['F', 'R', 'G', 'A', 'S'])
df['Count'] = [27, 2, 12, 2, 23, 4, 7, 1, 
               394, 32, 142, 19, 421, 38, 94, 11,
               18, 1, 13, 1, 24, 0, 4, 3,
               48, 6, 25, 4, 55, 15, 13, 4]
df[['F', 'R', 'G', 'A', 'S']] = df[['F', 'R', 'G', 'A', 'S']].astype('category')
df.head()

,F,R,G,A,S,Count
0,Both parents,B,M,1,N,27
1,Both parents,B,M,1,S,2
2,Both parents,B,M,2,N,12
3,Both parents,B,M,2,S,2
4,Both parents,B,F,1,N,23


In [7]:
fm = glm('Count ~ F*R*G*A*S', data=df, family=Poisson()).fit()

print(drop1_glm(fm))

          Df      Deviance         AIC       LRT  Pr(>Chi)
<none>        4.691557e-09  201.681384                    
F:R:G:A:S  1  3.638905e+00  203.320289  3.638905  0.056444


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


In [8]:
fm = update_glm(fm, drop='F:R:G:A:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      3.638905  203.320289                    
F:R:G:A  1  3.708219  201.389604  0.069314  0.792338
F:R:G:S  1  3.844518  201.525902  0.205612  0.650228
F:R:A:S  1  6.270808  203.952192  2.631903  0.104736
F:G:A:S  1  4.264350  201.945734  0.625445  0.429031
R:G:A:S  1  3.991229  201.672613  0.352324  0.552801


In [9]:
fm = update_glm(fm, drop='F:R:G:A')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      3.708219  201.389604                    
F:R:G:S  1  3.988333  199.669717  0.280114  0.596627
F:R:A:S  1  6.272211  201.953595  2.563992  0.109322
F:G:A:S  1  4.372985  200.054369  0.664765  0.414883
R:G:A:S  1  4.086900  199.768284   0.37868   0.53831


In [10]:
fm = update_glm(fm, drop='F:R:G:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      3.988333  199.669717                    
F:R:G    1  4.093347  197.774731  0.105014  0.745894
F:R:A:S  1  6.685475  200.366860  2.697142  0.100528
F:G:A:S  1  4.502332  198.183717  0.513999  0.473413
R:G:A:S  1  4.199296  197.880680  0.210963  0.646014


In [11]:
fm = update_glm(fm, drop='F:R:G')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      4.093347  197.774731                    
F:R:A:S  1  6.833659  198.515043  2.740312  0.097846
F:G:A:S  1  4.639732  196.321117  0.546386  0.459798
R:G:A:S  1  4.342070  196.023455  0.248723  0.617975


In [12]:
fm = update_glm(fm, drop='R:G:A:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      4.342070  196.023455                    
R:G:A    1  4.409247  194.090632  0.067177  0.795492
R:G:S    1  4.657215  194.338600  0.315145   0.57454
F:R:A:S  1  7.349332  197.030717  3.007262  0.082892
F:G:A:S  1  5.030993  194.712378  0.688923  0.406531


In [13]:
fm = update_glm(fm, drop='R:G:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      4.657215  194.338600                    
R:G:A    1  4.691905  192.373289   0.03469  0.852247
F:R:A:S  1  7.711777  195.393162  3.054562  0.080511
F:G:A:S  1  5.537633  193.219018  0.880418  0.348087


In [14]:
fm = update_glm(fm, drop='F:G:A:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      5.537633  193.219018                    
F:G:A    1  6.603769  192.285153  1.066136   0.30182
R:G:A    1  5.582112  191.263496  0.044479  0.832966
F:G:S    1  8.123313  193.804697   2.58568  0.107834
G:A:S    1  5.591351  191.272735  0.053717  0.816717
F:R:A:S  1  8.541894  194.223278  3.004261  0.083046


In [15]:
fm = update_glm(fm, drop='G:A:S')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      5.591351  191.272735                    
F:G:A    1  6.692369  190.373754  1.101019  0.294043
R:G:A    1  5.636214  189.317599  0.044864  0.832255
F:G:S    1  8.140539  191.821923  2.549188  0.110351
F:R:A:S  1  8.595396  192.276780  3.004045  0.083057


In [16]:
fm = update_glm(fm, drop='F:G:A')

print(drop1_glm(fm))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      6.692369  190.373754                    
R:G:A    1  6.955171  188.636556  0.262802  0.608202
F:G:S    1  9.026551  190.707936  2.334182  0.126561
F:R:A:S  1  9.746061  191.427446  3.053692  0.080554


In [17]:
fm = update_glm(fm, drop='F:G:S')

print(drop1_glm(fm))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>       9.026551  190.707936                    
F:G      1   9.565444  189.246828  0.538893  0.462892
G:S      1  10.614407  190.295792  1.587856  0.207633
R:G:A    1   9.191102  188.872486   0.16455  0.685001
F:R:A:S  1  11.999639  191.681023  2.973087   0.08466


In [18]:
fm = update_glm(fm, drop='F:G')

print(drop1_glm(fm))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>       9.565444  189.246828                    
G:S      1  11.292980  188.974364  1.727536  0.188726
R:G:A    1   9.724144  187.405528    0.1587  0.690357
F:R:A:S  1  12.537681  190.219065  2.972237  0.084705


In [19]:
fm = update_glm(fm, drop='G:S')

print(drop1_glm(fm))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      11.292980  188.974364                    
R:G:A    1  11.424154  187.105539  0.131175  0.717217
F:R:A:S  1  14.265216  189.946601  2.972237  0.084705


In [20]:
fm = update_glm(fm, drop='F:R:A:S')

print(drop1_glm(fm))

       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>     14.265216  189.946601                    
F:R:A   1  14.300729  187.982113  0.035512  0.850526
R:G:A   1  14.396391  188.077775  0.131175  0.717217
F:R:S   1  18.196847  191.878231   3.93163  0.047386
F:A:S   1  14.268519  187.949904  0.003303   0.95417
R:A:S   1  15.409442  189.090826  1.144225  0.284761


In [21]:
fm = update_glm(fm, drop='F:A:S')

print(drop1_glm(fm))

       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>     14.268519  187.949904                    
F:R:A   1  14.305619  185.987004    0.0371  0.847262
R:G:A   1  14.399694  186.081078  0.131175  0.717217
F:R:S   1  18.298268  189.979652  4.029748  0.044705
R:A:S   1  15.414132  187.095516  1.145613  0.284469


In [22]:
fm = update_glm(fm, drop='F:R:A')

print(drop1_glm(fm))

       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>     14.305619  185.987004                    
F:A     1  15.703296  185.384680  1.397677  0.237113
R:G:A   1  14.436794  184.118178  0.131175  0.717217
F:R:S   1  18.301762  187.983147  3.996143  0.045605
R:A:S   1  15.433464  185.114849  1.127845  0.288235


In [23]:
fm = update_glm(fm, drop='R:A:S')

print(drop1_glm(fm))

       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>     15.433464  185.114849                    
F:A     1  16.688557  184.369941  1.255092  0.262582
A:S     1  19.338734  187.020118   3.90527  0.048135
R:G:A   1  15.564639  183.246023  0.131175  0.717217
F:R:S   1  19.248028  186.929412  3.814564  0.050809


In [24]:
fm = update_glm(fm, drop='F:A')

print(drop1_glm(fm))

       Df   Deviance         AIC       LRT  Pr(>Chi)
<none>     16.688557  184.369941                    
A:S     1  20.937546  186.618931   4.24899  0.039274
R:G:A   1  16.819731  182.501116  0.131175  0.717217
F:R:S   1  20.492670  186.174054  3.804113  0.051127


In [25]:
fm = update_glm(fm, drop='F:R:S')

print(drop1_glm(fm))

       Df   Deviance         AIC        LRT  Pr(>Chi)
<none>     20.492670  186.174054                     
F:R     1  97.428375  261.109759  76.935705       0.0
F:S     1  27.402014  191.083398   6.909344  0.008575
R:S     1  21.062605  184.743989   0.569935  0.450285
A:S     1  24.741660  188.423044    4.24899  0.039274
R:G:A   1  20.623844  184.305229   0.131175  0.717217


In [26]:
fm = update_glm(fm, drop='R:S')

print(drop1_glm(fm))

       Df   Deviance         AIC        LRT  Pr(>Chi)
<none>     21.062605  184.743989                     
F:R     1  97.432132  259.113516  76.369527       0.0
F:S     1  27.405771  189.087155   6.343166  0.011783
A:S     1  25.177468  186.858852   4.114863  0.042508
R:G:A   1  21.193779  182.875164   0.131175  0.717217


In [27]:
print(fm.summary2())

                   Results: Generalized linear model
Model:                   GLM                 Method:            IRLS    
Model Family:            Poisson             AIC:               184.7440
Link Function:           Log                 BIC:               203.7986
Dependent Variable:      Count               Log-Likelihood:    -79.372 
Date:                    2026-10-27 13:41    LL-Null:           -1803.2 
No. Observations:        32                  Deviance:          21.063  
Df Model:                12                  Pearson chi2:      19.2    
Df Residuals:            19                  Scale:             1.0000  
------------------------------------------------------------------------
                         Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
------------------------------------------------------------------------
Intercept                3.2521   0.1595  20.3837 0.0000  2.9394  3.5649
F[T.Mother only]        -0.2626   0.1711  -1.5345 0.1249 -0.5979  0.072

In [28]:
print(chi2.sf(fm.deviance, fm.df_resid))

0.33335198490933027


In [29]:
np.exp(0.5520)

np.float64(1.736722992721326)

#### (b)

In [30]:
fm2 = glm('Count ~ F*R*G*A*S', data=df, family=Poisson()).fit()

print(drop1_glm(fm2))

          Df      Deviance         AIC       LRT  Pr(>Chi)
<none>        4.691557e-09  201.681384                    
F:R:G:A:S  1  3.638905e+00  203.320289  3.638905  0.056444


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


In [31]:
fm2 = update_glm(fm2, drop='F:R:G:A:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      3.638905  203.320289                    
F:R:G:A  1  3.708219  201.389604  0.069314  0.792338
F:R:G:S  1  3.844518  201.525902  0.205612  0.650228
F:R:A:S  1  6.270808  203.952192  2.631903  0.104736
F:G:A:S  1  4.264350  201.945734  0.625445  0.429031
R:G:A:S  1  3.991229  201.672613  0.352324  0.552801


In [32]:
fm2 = update_glm(fm2, drop='F:R:G:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      3.844518  201.525902                    
F:R:G:A  1  3.988333  199.669717  0.143815  0.704517
F:R:A:S  1  6.667043  202.348428  2.822526   0.09295
F:G:A:S  1  4.339614  200.020999  0.495097  0.481663
R:G:A:S  1  4.056734  199.738118  0.212216  0.645036


In [33]:
fm2 = update_glm(fm2, drop='R:G:A:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      4.056734  199.738118                    
R:G:S    1  4.442341  198.123725  0.385607  0.534618
F:R:G:A  1  4.199296  197.880680  0.142562  0.705747
F:R:A:S  1  7.168104  200.849488   3.11137  0.077748
F:G:A:S  1  4.665529  198.346914  0.608795  0.435242


In [34]:
fm2 = update_glm(fm2, drop='R:G:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      4.442341  198.123725                    
F:R:G:A  1  4.538320  196.219704  0.095979  0.756709
F:R:A:S  1  7.550684  199.232069  3.108344  0.077892
F:G:A:S  1  5.261367  196.942751  0.819026  0.365465


In [35]:
fm2 = update_glm(fm2, drop='F:G:A:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      5.261367  196.942751                    
F:G:S    1  7.992275  197.673659  2.730908  0.098423
G:A:S    1  5.327200  195.008584  0.065833  0.797503
F:R:G:A  1  5.368691  195.050075  0.107324  0.743212
F:R:A:S  1  8.343346  198.024730  3.081979  0.079164


In [36]:
fm2 = update_glm(fm2, drop='G:A:S')

print(drop1_glm(fm2))

        Df  Deviance         AIC       LRT  Pr(>Chi)
<none>      5.327200  195.008584                    
F:G:S    1  8.013994  195.695378  2.686794  0.101183
F:R:G:A  1  5.430203  193.111587  0.103003  0.748256
F:R:A:S  1  8.401719  196.083103  3.074519  0.079528


In [37]:
fm2 = update_glm(fm2, drop='F:G:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>       8.013994  195.695378                    
G:S      1   9.660778  195.342162  1.646784  0.199397
F:R:G:A  1   8.061357  193.742741  0.047363  0.827717
F:R:A:S  1  11.009366  196.690750  2.995372  0.083503


In [38]:
fm2 = update_glm(fm2, drop='G:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>       9.660778  195.342162                    
F:R:G:A  1   9.690705  193.372090  0.029927  0.862655
F:R:A:S  1  12.633015  196.314399  2.972237  0.084705


In [39]:
fm2 = update_glm(fm2, drop='F:R:A:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      12.633015  196.314399                    
F:R:S    1  16.564645  198.246029   3.93163  0.047386
F:A:S    1  12.636317  194.317702  0.003303   0.95417
R:A:S    1  13.777240  195.458624  1.144225  0.284761
F:R:G:A  1  12.662942  194.344326  0.029927  0.862655


In [40]:
fm2 = update_glm(fm2, drop='F:A:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      12.636317  194.317702                    
F:R:S    1  16.666066  196.347450  4.029748  0.044705
R:A:S    1  13.781930  193.463314  1.145613  0.284469
F:R:G:A  1  12.666245  192.347629  0.029927  0.862655


In [41]:
fm2 = update_glm(fm2, drop='R:A:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      13.781930  193.463314                    
A:S      1  17.706227  195.387611  3.924297  0.047593
F:R:S    1  17.615097  195.296481  3.833166  0.050248
F:R:G:A  1  13.811858  191.493242  0.029927  0.862655


In [42]:
fm2 = update_glm(fm2, drop='F:R:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      17.615097  195.296481                    
F:S      1  24.170695  199.852079  6.555598  0.010455
R:S      1  18.156456  193.837840  0.541359   0.46187
A:S      1  21.510340  197.191724  3.895244  0.048423
F:R:G:A  1  17.645024  193.326408  0.029927  0.862655


In [43]:
fm2 = update_glm(fm2, drop='R:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      18.156456  193.837840                    
F:S      1  24.174452  197.855836  6.017996  0.014161
A:S      1  21.946148  195.627532  3.789692  0.051569
F:R:G:A  1  18.186383  191.867768  0.029927  0.862655


In [44]:
fm2 = update_glm(fm2, drop='A:S')

print(drop1_glm(fm2))

        Df   Deviance         AIC       LRT  Pr(>Chi)
<none>      21.946148  195.627532                    
F:S      1  28.420961  200.102346  6.474813  0.010941
F:R:G:A  1  21.976075  193.657460  0.029927  0.862655


In [45]:
print(fm2.summary2())

                          Results: Generalized linear model
Model:                       GLM                      Method:                 IRLS    
Model Family:                Poisson                  AIC:                    195.6275
Link Function:               Log                      BIC:                    222.0108
Dependent Variable:          Count                    Log-Likelihood:         -79.814 
Date:                        2026-10-27 13:41         LL-Null:                -1803.2 
No. Observations:            32                       Deviance:               21.946  
Df Model:                    17                       Pearson chi2:           20.0    
Df Residuals:                14                       Scale:                  1.0000  
--------------------------------------------------------------------------------------
                                       Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
----------------------------------------------------------------------

In [46]:
print(chi2.sf(fm2.deviance, fm2.df_resid))

0.07972767576835037


In [47]:
df['fitted_fm2'] = fm2.fittedvalues 
collapsed_fitted2 = df.pivot_table(index=['F'], columns='S', values='fitted_fm2', aggfunc='sum').sort_index()
print(collapsed_fitted2)

S                  N      S
F                          
Both parents  1120.0  109.0
Mother only    200.0   34.0


In [48]:
np.exp(0.5578), 1120*34/(200*109)

(np.float64(1.7468252543177483), 1.7467889908256882)